In [1]:
import pandas as pd

df = pd.read_csv("data/superstore_sales.csv")
df.head()

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,State,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales
0,1,CA-2017-152156,08/11/2017,11/11/2017,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420.0,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600
1,2,CA-2017-152156,08/11/2017,11/11/2017,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420.0,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400
2,3,CA-2017-138688,12/06/2017,16/06/2017,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,California,90036.0,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200
3,4,US-2016-108966,11/10/2016,18/10/2016,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311.0,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775
4,5,US-2016-108966,11/10/2016,18/10/2016,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311.0,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680


In [2]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9800 entries, 0 to 9799
Data columns (total 18 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Row ID         9800 non-null   int64  
 1   Order ID       9800 non-null   object 
 2   Order Date     9800 non-null   object 
 3   Ship Date      9800 non-null   object 
 4   Ship Mode      9800 non-null   object 
 5   Customer ID    9800 non-null   object 
 6   Customer Name  9800 non-null   object 
 7   Segment        9800 non-null   object 
 8   Country        9800 non-null   object 
 9   City           9800 non-null   object 
 10  State          9800 non-null   object 
 11  Postal Code    9789 non-null   float64
 12  Region         9800 non-null   object 
 13  Product ID     9800 non-null   object 
 14  Category       9800 non-null   object 
 15  Sub-Category   9800 non-null   object 
 16  Product Name   9800 non-null   object 
 17  Sales          9800 non-null   float64
dtypes: float

In [3]:
df["Order Date"] = pd.to_datetime(df["Order Date"], dayfirst=True)
df["Ship Date"] = pd.to_datetime(df["Ship Date"], dayfirst=True)

print("First order:", df["Order Date"].min())
print("Last order:", df["Order Date"].max())

First order: 2015-01-03 00:00:00
Last order: 2018-12-30 00:00:00


In [4]:
(df["Ship Date"] < df["Order Date"]).sum()

np.int64(0)

In [5]:
print("Duplicate rows:", df.duplicated().sum())
df[df["Postal Code"].isna()][["City", "State"]].drop_duplicates()

Duplicate rows: 0


,City,State
2234,Burlington,Vermont


In [6]:
snapshot = df["Order Date"].max() + pd.Timedelta(days=1)
snapshot

Timestamp('2018-12-31 00:00:00')

In [7]:
rfm = df.groupby("Customer ID").agg(
    LastOrder=("Order Date", "max"),
    Frequency=("Order ID", "nunique"),
    Monetary=("Sales", "sum"),
)

rfm["Recency"] = (snapshot - rfm["LastOrder"]).dt.days

print(rfm.shape)
rfm.head()

(793, 4)


,LastOrder,Frequency,Monetary,Recency
Customer ID,,,,
AA-10315,2018-06-29,5,5563.560,185
AA-10375,2018-12-11,9,1056.390,20
AA-10480,2018-04-15,4,1790.512,260
AA-10645,2018-11-05,6,5086.935,56
AB-10015,2017-11-10,3,886.156,416


In [8]:
old_way = df.groupby("Customer ID")["Order ID"].count()
print("Old average frequency (counting rows):", round(old_way.mean(), 2))
print("New average frequency (unique orders):", round(rfm["Frequency"].mean(), 2))
print("Total revenue:", round(rfm["Monetary"].sum(), 2))

Old average frequency (counting rows): 12.36
New average frequency (unique orders): 6.21
Total revenue: 2261536.78
